# YOLOv8n-seg training and integer quantization for STM32N6

This notebook describes the preparation, training, quantization, and evaluation of a YOLOv8n instance segmentation model for the NUCLEO-N657X0-Q. The reference configuration uses a 256 × 256 RGB input and the two-output segmentation interface implemented by the repository's C postprocessor. A preliminary inference with the bundled COCO model establishes the expected tensor structure before a custom dataset is introduced.

The procedure produces a validation-selected PyTorch checkpoint, a fully quantized TFLite model, and a JSON description of the class order and tensor interface. These artifacts form the input to the compilation and deployment procedure in [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb). The [application architecture](../Doc/Application-Overview.md) describes their relationship to the camera pipeline, Neural-ART runtime, and firmware.

## Software environment and execution order

Reference inference requires `ai-edge-litert`, NumPy, and Pillow in the Jupyter environment. The TensorFlow Lite interpreter is supported as an alternative. Training and export use a separate Python 3.10 or 3.11 environment. The dependency versions below follow the Ultralytics 8.3.88 workflow documented by ST. The exporter may require additional conversion packages. A record of the resolved environment can be obtained with `python -m pip freeze` after a successful export. [ST segmentation export procedure](https://github.com/STMicroelectronics/stm32ai-modelzoo-services/blob/main/instance_segmentation/docs/tuto/How_to_deploy_yolov8_instance_segmentation.md)

```bash
python -m pip install "ultralytics==8.3.88" "tensorflow==2.15.0" "onnx==1.15.0" "onnxruntime==1.18.1" "keras==2.15.0" "tf-keras==2.15.0" ipykernel
python -m ipykernel install --user --name stm32n6-yolov8 --display-name "Python 3 (STM32N6 YOLOv8)"
```

Cells execute sequentially using the selected Jupyter kernel. Sections 1 and 1.1 operate on the bundled model and require no custom dataset. Subsequent sections require the labeled image directories specified in the configuration cell. Export can also run under WSL2 when the Windows conversion environment presents dependency incompatibilities. The exported TFLite and JSON files can then be transferred to the workstation running ST Edge AI Core.

In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "Model").is_dir() else HERE.parent
# REPO = Path(r"C:\projects\STM32N6-Edge_Computer_Vision")
BASELINE = REPO / "Model" / "yolov8n_256_quant_pc_ii_seg_coco-st.tflite"
assert BASELINE.is_file(), f"Set REPO to the repository root. Missing {BASELINE}"

# Training artifacts and datasets reside beside the repository, independently of source updates.
WORK = REPO.parent / "stm32n6_model_work"
DATASET_ROOT = REPO.parent / "insect_dataset"
CLASS_NAMES = ["insect"]  # Index order corresponds to the class IDs in the annotation files.
# Reference input size determines detection grids, activation memory, and decoder dimensions.
IMAGE_SIZE = 256
IMAGE = None  # Optional: Path(r"C:\path\capture.jpg")

## 1. Reference inference and input representation

The bundled COCO TFLite model accepts an NHWC tensor with shape `[1, 256, 256, 3]`, where the dimensions represent batch, height, width, and RGB channels. Its input uses signed 8-bit integers, a scale of approximately 1/255, and a zero point of −128. ST Edge AI Core generates an unsigned input interface with the same scale and a zero point of 0 for the firmware.

Affine dequantization recovers a real value using `real = scale × (quantized - zero_point)`. For an RGB channel byte of 128, the corresponding normalized value is approximately 0.502. The host TFLite represents this value as signed integer 0. The generated firmware interface represents it as unsigned integer 128. Both encodings therefore describe the same normalized channel value.

The inference helper normalizes RGB bytes to the model's real input range and applies the inverse affine transformation. Rounding and clipping precede conversion to the integer dtype to prevent wraparound. The returned arrays retain their quantized representation for subsequent inspection.

With `IMAGE = None`, a uniform input checks model loading and execution. A COCO object image is required to examine meaningful object predictions. Image resizing preserves the firmware inference pipe's aspect-ratio transformation, although differences in interpolation and ISP processing may alter individual pixels. Numerical comparisons between PC and board require identical input RGB bytes.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite import Interpreter

def open_tflite(path):
    interpreter = Interpreter(model_path=str(path), num_threads=2)
    # Tensor allocation establishes the buffers required for input assignment and inference.
    interpreter.allocate_tensors()
    return interpreter

def tensor_info(detail):
    scale, zero_point = detail["quantization"]
    return {
        "name": detail["name"],
        "shape": detail["shape"].tolist(),
        "dtype": np.dtype(detail["dtype"]).name,
        "scale": float(scale),
        "zero_point": int(zero_point),
    }

def run_tflite(interpreter, rgb):
    inputs = interpreter.get_input_details()
    assert len(inputs) == 1, "Expected a single RGB input."
    info = inputs[0]
    scale, zero_point = info["quantization"]
    assert scale > 0 and info["dtype"] in (np.int8, np.uint8)
    limits = np.iinfo(info["dtype"])
    # The leading axis is the batch. Division by 255 recovers the real-valued RGB range used by this graph.
    real = rgb[None].astype(np.float32) / 255
    assert real.shape == tuple(info["shape"])
    # Invert the affine dequantization formula. Round, then clamp before casting to prevent integer wraparound.
    data = np.clip(np.rint(real / scale + zero_point), limits.min, limits.max)
    interpreter.set_tensor(info["index"], data.astype(info["dtype"]))
    interpreter.invoke()
    # Raw outputs retain their individual shapes and affine encodings for subsequent decoding.
    return [interpreter.get_tensor(d["index"]) for d in interpreter.get_output_details()]

baseline = open_tflite(BASELINE)
for detail in [*baseline.get_input_details(), *baseline.get_output_details()]:
    print(tensor_info(detail))

if IMAGE is None:
    rgb = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 220, dtype=np.uint8)
else:
    reference = Image.open(IMAGE).convert("RGB")
    rgb = np.asarray(reference.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))

raw_outputs = run_tflite(baseline, rgb)
print("COCO outputs:", [(output.shape, output.dtype.name) for output in raw_outputs])

### 1.1. Candidate decoding and prototype mask reconstruction

The detection tensor contains 116 attribute rows and 1,344 candidate columns. Each candidate comprises four box values, 80 class scores, and 32 mask coefficients. The indexing below follows the attribute-first layout consumed by the C decoder.

The example selects the highest class score across all candidates and reports its class ID, normalized box, and mask coefficients. This selection precedes confidence filtering and non-maximum suppression (NMS), so a candidate is returned even for an empty image.

Instance segmentation represents each object's mask as a combination of shared prototype maps. At each position in the 64 × 64 prototype grid, a dot product combines 32 prototype values with the selected candidate's 32 coefficients. The example performs this calculation on dequantized floating-point values and applies the bundled C decoder's threshold of 0.5. The firmware performs integer accumulation with a threshold scaled by the two quantization factors. Rounding can introduce differences near the threshold. The displayed mask precedes box clipping, confidence filtering, and NMS.

In [ ]:
detection_details = baseline.get_output_details()[0]
assert tuple(detection_details["shape"]) == (1, 116, 1344)
detection_scale, detection_zero_point = detection_details["quantization"]

# Convert to a wider type before subtraction so int8 values cannot overflow.
detections = (raw_outputs[0][0].astype(np.float32) - detection_zero_point) * detection_scale
class_scores = detections[4:84, :]
class_id, candidate_id = np.unravel_index(class_scores.argmax(), class_scores.shape)
print("Candidate and class ID:", candidate_id, class_id)
print("Class score before thresholding or NMS:", float(class_scores[class_id, candidate_id]))
print("Normalized cx, cy, width, height:", detections[:4, candidate_id].tolist())
coefficients = detections[84:, candidate_id]
print("Mask coefficients for this candidate:", coefficients.shape[0])

prototype_details = baseline.get_output_details()[1]
assert tuple(prototype_details["shape"]) == (1, 64, 64, 32)
prototype_scale, prototype_zero_point = prototype_details["quantization"]
prototypes = (raw_outputs[1][0].astype(np.float32) - prototype_zero_point) * prototype_scale

# Multiply each pixel's 32 prototype values by the same 32 candidate coefficients.
# The dot product collapses the channel axis, yielding one value per mask pixel.
mask_values = prototypes @ coefficients
prototype_mask = (mask_values >= 0.5).astype(np.uint8) * 255
display(Image.fromarray(prototype_mask).resize((256, 256), Image.Resampling.NEAREST))

## 2. Dataset structure and annotation validation

The dataset consists of paired image and polygon-label directories under `insect_dataset/`, beside the repository. Each split uses `images/{train,val,test}` and `labels/{train,val,test}`. An image such as `001.jpg` corresponds to `001.txt` at the same relative location in the label directory.

Each annotation row contains one class ID followed by at least three coordinate pairs. Coordinates are normalized independently by image width and height. For example, the following row describes one four-vertex polygon assigned to class 0.

```text
0 0.40 0.44 0.54 0.43 0.55 0.57 0.41 0.55
```

An empty TXT file explicitly identifies an image with no labeled objects. The validation cell checks label availability, class-index bounds, coordinate bounds, and polygon syntax. It reports the image, background, and instance counts for each split, then writes the dataset YAML used by Ultralytics. [Segmentation annotation format](https://docs.ultralytics.com/datasets/segment/)

The training split supplies examples for weight optimization. The validation split supports checkpoint selection and quantization calibration in this procedure. The test split provides held-out examples for the final comparison. Capture events or collection sessions should remain within a single split, because adjacent burst frames are strongly correlated. Empty backgrounds, reflections, motion, and variations in illumination represent relevant operating conditions for a trap camera.

Annotation validity also depends on spatial alignment and class assignment. The polygon preview supports manual inspection of these properties and of the object's apparent size at 256 × 256 resolution.

In [ ]:
import yaml

assert CLASS_NAMES and len(CLASS_NAMES) == len(set(CLASS_NAMES))
assert len(CLASS_NAMES) <= 255, "The demo postprocessor uses 8-bit class IDs."
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
first_test_image = None
training_example = None

for split in ("train", "val", "test"):
    images_root = DATASET_ROOT / "images" / split
    # Mirror subdirectories too, so images with the same stem in different folders keep distinct labels.
    labels_root = DATASET_ROOT / "labels" / split
    images = sorted(p for p in images_root.rglob("*") if p.suffix.lower() in image_extensions)
    assert images, f"No images in {images_root}"
    instances = 0
    backgrounds = 0
    for image_path in images:
        label = labels_root / image_path.relative_to(images_root).with_suffix(".txt")
        assert label.is_file(), f"Missing {label}. Use an empty file for a background image."
        annotation = label.read_text(encoding="utf-8")
        backgrounds += not annotation.strip()
        for row_number, row in enumerate(annotation.splitlines(), 1):
            tokens = row.split()
            if not tokens:
                continue
            location = f"{label} line {row_number}"
            # One class ID followed by at least three complete (x, y) pairs gives at least seven tokens.
            assert len(tokens) >= 7 and len(tokens) % 2 == 1, f"Invalid polygon: {location}"
            assert 0 <= int(tokens[0]) < len(CLASS_NAMES), f"Invalid class ID: {location}"
            assert all(0 <= float(x) <= 1 for x in tokens[1:]), f"Invalid coordinates: {location}"
            if split == "train" and training_example is None:
                training_example = (image_path, label)
            instances += 1
    assert instances, f"The {split} split has no labeled instances."
    print(f"{split}: {len(images)} images, {backgrounds} backgrounds, {instances} instances")
    if split == "test":
        first_test_image = images[0]

# Absolute dataset root plus relative split folders lets Ultralytics find the same data from any working directory.
WORK.mkdir(exist_ok=True)
DATA_YAML = WORK / "insects.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": dict(enumerate(CLASS_NAMES)),
}, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

### 2.1. Annotation visualization

The visualization overlays the first annotated training example with its polygon boundaries and resizes the result to the model input dimensions. This inspection complements the syntax checks by exposing displaced polygons, incorrect class assignments, and objects that occupy few pixels after downsampling. A dataset review should include representative examples from every split.

In [ ]:
from PIL import ImageDraw

image_path, positive_label = training_example
preview = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(preview)
for row in positive_label.read_text(encoding="utf-8").splitlines():
    values = row.split()
    if values:
        class_id, coordinates = int(values[0]), list(map(float, values[1:]))
        # YOLO stores relative positions. Convert them back to pixel coordinates for PIL.
        pixels = [(x * (preview.width - 1), y * (preview.height - 1))
                  for x, y in zip(coordinates[::2], coordinates[1::2])]
        draw.polygon(pixels, outline="lime", width=2)
        print("Label:", CLASS_NAMES[class_id])
display(preview.copy().resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))

## 3. Transfer learning and checkpoint selection

Transfer learning initializes the network from pretrained YOLOv8n-seg weights and optimizes its parameters using the labeled dataset. Existing visual features provide the starting representation, while the task head adapts to the configured class set.

The initial configuration retains the 256 × 256 input used by the reference firmware. An epoch represents one pass through the training dataset. The 50-epoch setting defines the initial training duration and requires assessment through the recorded training and validation curves. Improving training performance accompanied by deteriorating validation performance indicates overfitting.

Ultralytics stores the run configuration, curves, and weights in the training output directory. `best.pt` denotes the checkpoint selected by validation fitness. `last.pt` contains the latest checkpoint and supports resuming an interrupted run. The code obtains the actual output path from the trainer, accounting for run-directory suffixes.

Training arguments can also be supplied through a YAML file. The [configuration reference](../Doc/Source-Walkthrough.md#3-model-and-training-configuration) describes the dataset, architecture, and training configurations, including precedence rules. The final statement prints the location of the run's saved `args.yaml`.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
DEVICE = None  # Use 0 for the first CUDA GPU or "cpu" to force CPU.
# Pretrained segmentation weights initialize the visual features for transfer learning.
trainer = YOLO("yolov8n-seg.pt")
# Input dimensions match the firmware. workers=0 avoids subprocess spawning in Windows Jupyter.
trainer.train(
    data=str(DATA_YAML), imgsz=IMAGE_SIZE, epochs=EPOCHS,
    project=str(WORK / "runs"), name="insect_yolov8n_seg",
    device=DEVICE, workers=0, seed=42,
)
# The trainer provides the resolved run path, including any automatically added suffix.
BEST_PT = Path(trainer.trainer.best).resolve()
assert BEST_PT.is_file(), BEST_PT
print(BEST_PT)
print("Saved training arguments:", BEST_PT.parent.parent / "args.yaml")

## 4. Full-integer TFLite export

Full-integer quantization encodes learned weights and intermediate activations with low-precision values. Calibration estimates activation ranges from representative inputs and determines their scales and zero points. Per-channel quantization allows supported weight channels to use separate scales, while the input and output interfaces inspected here use scalar affine parameters.

The export procedure follows ST's workflow for Ultralytics 8.3.88. In `ultralytics/engine/exporter.py`, the `onnx2tf.convert` argument is set to `quant_type="per-channel"`. Following this dependency change, the Jupyter kernel requires a restart and the configuration, reference, and dataset cells require re-execution. An existing checkpoint can be assigned to `BEST_PT` in the export cell.

Calibration uses the validation split defined in `insects.yaml`. Its images should represent the camera's expected backgrounds and illumination. The export retains raw detection and prototype outputs because filtering and mask reconstruction are implemented in the C postprocessor.

The exporter can produce several TFLite variants. This procedure selects `*_full_integer_quant.tflite`, verifies the signed 8-bit interfaces and expected tensor shapes, and records the class order in `insect_model.json`. Structural agreement must be followed by prediction validation, since equal shapes alone do not establish equivalent output semantics.

In [ ]:
import inspect
from ultralytics import YOLO  # Explicit import supports export after a kernel restart.
from ultralytics.engine import exporter

assert 'quant_type="per-channel"' in inspect.getsource(exporter), (
    f"Set per-channel quantization in {inspect.getfile(exporter)} and restart the kernel."
)

# An existing training checkpoint can be assigned here after a kernel restart.
# BEST_PT = WORK / "runs" / "insect_yolov8n_seg" / "weights" / "best.pt"
fp_model = YOLO(str(BEST_PT))
assert list(fp_model.names.values()) == CLASS_NAMES
# Validation images provide calibration data. Batch 1 matches capture, and NMS remains in C.
exported = Path(fp_model.export(
    format="tflite", imgsz=IMAGE_SIZE, batch=1, int8=True, nms=False,
    data=str(DATA_YAML), split="val", device="cpu",
)).resolve()
# The full_integer_quant artifact exposes integer input and output interfaces.
QUANTIZED = exported.parent / f"{BEST_PT.stem}_full_integer_quant.tflite"
assert QUANTIZED.is_file(), f"Look for the full_integer_quant file beside {exported}"

In [ ]:
candidate = open_tflite(QUANTIZED)
input_specs = [tensor_info(d) for d in candidate.get_input_details()]
output_specs = [tensor_info(d) for d in candidate.get_output_details()]
print(json.dumps({"input": input_specs, "outputs": output_specs}, indent=2))
assert len(input_specs) == 1 and input_specs[0]["shape"] == [1, 256, 256, 3]
assert input_specs[0]["dtype"] == "int8"
assert all(spec["dtype"] == "int8" and spec["scale"] > 0 for spec in output_specs)
assert len(output_specs) == 2
# At 256 × 256 the head uses grids 32² + 16² + 8² = 1344. Each row stores box, classes, and mask coefficients.
expected_shapes = {(1, 4 + len(CLASS_NAMES) + 32, 1344), (1, 64, 64, 32)}
assert {tuple(spec["shape"]) for spec in output_specs} == expected_shapes, (
    "Tensor shapes differ from the demo. Check the export and C integration."
)

# The manifest records class order and the TFLite interface for the deployment procedure.
manifest = {
    "model": str(QUANTIZED),
    "class_names": CLASS_NAMES,
    "imgsz": IMAGE_SIZE,
    "input": input_specs,
    "outputs": output_specs,
}
MANIFEST = WORK / "insect_model.json"
MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(QUANTIZED)
print(MANIFEST)

## 5. Evaluation of floating-point and quantized models

The trained PyTorch checkpoint and the quantized TFLite model are evaluated on the same held-out test split and input resolution. Box and mask metrics quantify any loss of prediction quality introduced by export and quantization. Ultralytics supplies the preprocessing and validation settings for these dataset-level measurements.

Precision expresses the fraction of reported objects that are correct. Recall expresses the fraction of annotated objects that are recovered. Intersection over union (IoU) measures spatial overlap as intersection area divided by union area. The reported `mAP50–95` averages average precision across IoU thresholds from 0.50 to 0.95 and across evaluated classes. Box mAP evaluates rectangular regions, while mask mAP evaluates object masks.

The image preview applies the firmware's confidence threshold of 0.5, NMS IoU threshold of 0.4, and maximum of 10 objects. These prediction settings support visual comparison with the demo. Empty scenes and difficult insect captures provide additional evidence about false detections and missed objects. Final validation uses images and preprocessing representative of the deployed trap.

In [ ]:
models = {"PyTorch FP32": YOLO(str(BEST_PT)), "TFLite INT8": YOLO(str(QUANTIZED), task="segment")}
# Identical test data and resolution support comparison across the two model representations.
for name, model in models.items():
    metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=1, device="cpu")
    print(name, {"box mAP50-95": metrics.box.map, "mask mAP50-95": metrics.seg.map})

sample = Image.open(first_test_image if IMAGE is None else IMAGE).convert("RGB")
sample = sample.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
for name, model in models.items():
    result = model.predict(sample, imgsz=IMAGE_SIZE, conf=0.5, iou=0.4, max_det=10, device="cpu")[0]
    print(name)
    # Ultralytics plot() returns BGR. PIL expects RGB, so swap the red and blue channels.
    display(Image.fromarray(result.plot()[..., ::-1]))

The exported TFLite and `insect_model.json` provide the inputs to [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb). Transfer to another workstation requires updating the model path while retaining the training class order.